# AI Learning OS — RAG Baseline

A RAG pipeline for the AI Learning Chief of Staff. Given a user's learning goal and a
personal knowledge base of saved resources (videos, articles, courses, tutorials),
the system retrieves the most relevant items and generates a specific weekly action plan.

Functions follow the standard course pattern: `search` → `build_prompt` → `llm` → `rag`.

In [1]:
import json

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

## 1. Load Your Data

Replace this example with your own data. Each document should be a dict with a few text fields you want to search over.

In [2]:
with open("../data/resources.json") as f:
    documents = json.load(f)

print(f"Loaded {len(documents)} resources")
documents[0]

Loaded 25 resources


{'title': 'Neural Networks: Zero to Hero',
 'type': 'course',
 'topic': 'LLMs',
 'url': 'https://karpathy.ai/zero-to-hero.html',
 'description': "Andrej Karpathy's ground-up video series that builds neural networks from scratch in Python, culminating in a from-scratch GPT. Covers backpropagation, MLP, WaveNet, and the transformer architecture. Essential foundation for anyone who wants to truly understand how LLMs work rather than just use them.",
 'difficulty': 'intermediate'}

## 2. Build the Index

minsearch builds an in-memory TF-IDF index over the fields you specify.

In [3]:
index = Index(
    text_fields=["title", "description", "topic"],
    keyword_fields=["type", "difficulty"],
)

index.fit(documents)

## 3. Define the RAG Functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [4]:
def search(query):
    return index.search(query, num_results=5)

In [5]:
instructions = """
You are an AI Learning Chief of Staff. Your job is to help the user make focused,
structured progress toward their learning goal by drawing on their personal saved resources.

Using the CONTEXT below — resources retrieved from the user's knowledge base — give a
specific, actionable recommendation: name each resource, explain what it covers and why
it is relevant right now, and suggest a realistic order and time allocation for the week.
Keep your answer grounded in the provided resources.
""".strip()


def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [6]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [7]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

In [8]:
print(rag("I want to become job-ready in AI engineering in 4 months. I have 10 hours a week. What should I focus on this week?"))

To effectively become job-ready in AI engineering within your 4-month timeframe, focusing on a strong foundation in prompt engineering this week is essential. Given that you have 10 hours to dedicate, here's a structured plan:

### Learning Plan for the Week (10 hours total)

1. **Prompt Engineering Guide** (2 hours)
   - **Resource**: [Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering)
   - **Overview**: This guide covers essential strategies for crafting effective prompts, including clear instructions, task splitting, and systematic testing.
   - **Why It's Relevant**: As a beginner, mastering these foundational strategies is crucial for working with AI models effectively. Understanding the basics will set the stage for more complex topics ahead.

2. **Advanced Prompt Engineering: Chain of Thought and Self-Consistency** (3 hours)
   - **Resource**: [Advanced Prompt Engineering: Chain of Thought and Self-Consistency](https://www.youtube.com/watch?v=d